# Book recommendations with the Book-Crossing dataset

This notebook takes the three raw files through Bronze, Silver and Gold, then uses cosine similarity to find similar books and make recommendations for individual users. In Colab, run the first cell and upload `book-crossings.zip` when prompted.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import io

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, load_npz, save_npz
from sklearn.neighbors import NearestNeighbors
from IPython.display import display

pd.set_option("display.max_colwidth", None)

zip_path = Path("book-crossings.zip")
if not zip_path.exists():
    from google.colab import files
    files.upload()
if not zip_path.exists():
    raise FileNotFoundError("Upload the file named book-crossings.zip.")

for folder in ["bronze", "silver", "gold"]:
    Path(folder).mkdir(exist_ok=True)

## 1. Bronze

The files use semicolons and can be decoded with Latin-1. The Books file has some escaped quotes and semicolons inside titles. Its fields are consistently separated by `";"`, so I split on that boundary to keep those rows and preserve the text. I keep the source missing-value marker `NULL` and blank strings here and handle them in Silver.

In [2]:
source_files = {
    "users": "BX-Users.csv",
    "books": "BX-Books.csv",
    "ratings": "BX-Book-Ratings.csv",
}

timestamp = pd.Timestamp.now(tz="UTC")
with ZipFile(zip_path) as archive:
    for table, filename in source_files.items():
        with io.TextIOWrapper(archive.open(filename), encoding="latin-1") as source:
            if table == "books":
                rows = [line.rstrip("\r\n")[1:-1].split('";"') for line in source]
                header, *records = rows
                assert all(len(row) == len(header) for row in records)
                frame = pd.DataFrame(records, columns=header, dtype="string")
            else:
                frame = pd.read_csv(source, sep=";", keep_default_na=False,
                                    dtype={"ISBN": "string"})
        frame["ingestion_timestamp"] = timestamp
        frame.to_parquet(f"bronze/{table}.parquet", index=False)
        print(table, frame.shape)

users (278858, 4)
books (271379, 9)
ratings (1149780, 4)


## 2. Bronze exploration

In [3]:
raw_users = pd.read_parquet("bronze/users.parquet")
raw_books = pd.read_parquet("bronze/books.parquet")
raw_ratings = pd.read_parquet("bronze/ratings.parquet")

for name, frame in [("Users", raw_users), ("Books", raw_books), ("Ratings", raw_ratings)]:
    missing = frame.isna() | frame.isin(["", "NULL"])
    print(name, frame.shape)
    display(pd.DataFrame({"dtype": frame.dtypes.astype(str), "missing": missing.sum()}))

print("Rating distribution")
display(raw_ratings["Book-Rating"].value_counts().sort_index().rename("count").to_frame())

current_year = pd.Timestamp.today().year
ages = pd.to_numeric(raw_users["Age"], errors="coerce")
years = pd.to_numeric(raw_books["Year-Of-Publication"], errors="coerce")
print("Age and publication year distributions")
display(pd.DataFrame({"age": ages.describe(), "year": years.describe()}))
print("Ages outside 5–100:", int(((ages < 5) | (ages > 100)).sum()))
print("Zero or future years:", int(((years == 0) | (years > current_year)).sum()))

print("Rating records per user and per ISBN with at least one record")
counts = {"per user": raw_ratings.groupby("User-ID").size(),
          "per ISBN": raw_ratings.groupby("ISBN").size()}
display(pd.DataFrame({name: values.describe() for name, values in counts.items()}))

# Use catalog users and books as the denominator, and count distinct pairs in that catalog.
in_catalog = raw_ratings["User-ID"].isin(raw_users["User-ID"]) & raw_ratings["ISBN"].isin(raw_books["ISBN"])
pairs = raw_ratings.loc[in_catalog, ["User-ID", "ISBN"]].drop_duplicates()
coverage = 100 * len(pairs) / (raw_users["User-ID"].nunique() * raw_books["ISBN"].nunique())
print(f"Recorded pairs: {coverage:.6f}% (empty pairs: {100 - coverage:.6f}%)")

Users (278858, 4)
Books (271379, 9)
Ratings (1149780, 4)
Rating distribution
Age and publication year distributions
Ages outside 5–100: 1248
Zero or future years: 4630
Rating records per user and per ISBN with at least one record
Recorded pairs: 0.001363% (empty pairs: 99.998637%)


,dtype,missing
User-ID,int64,0
Location,object,0
Age,object,110762
ingestion_timestamp,"datetime64[us, UTC]",0


,dtype,missing
ISBN,string,0
Book-Title,string,0
Book-Author,string,0
Year-Of-Publication,string,0
Publisher,string,0
Image-URL-S,string,0
Image-URL-M,string,0
Image-URL-L,string,0
ingestion_timestamp,"datetime64[us, UTC]",0


,dtype,missing
User-ID,int64,0
ISBN,string,0
Book-Rating,int64,0
ingestion_timestamp,"datetime64[us, UTC]",0


,count
Book-Rating,
0,716109
1,1770
2,2759
3,5996
4,8904
5,50974
6,36924
7,76457
8,103736


,age,year
count,168096.000000,271379.0
mean,34.751434,1959.75605
std,14.428097,258.011363
min,0.000000,0.0
25%,24.000000,1989.0
50%,32.000000,1995.0
75%,44.000000,2000.0
max,244.000000,2050.0


,per user,per ISBN
count,105283.000000,340556.000000
mean,10.920851,3.376185
std,90.562825,12.436252
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,4.000000,2.000000
max,13602.000000,2502.000000


- **Shapes, types and missing data.** Bronze has 278,858 users, 271,379 books and 1,149,780 rating records. Age is stored as text and has 110,762 missing values. I will convert ages and publication years to numbers in Silver and fill missing ages with the median valid age.
- **Ratings.** There are 716,109 zero ratings. I will leave those out of the explicit-rating model and keep ratings from 1 to 10.
- **Ages and years.** There are 1,248 ages outside 5–100 and 4,630 publication years that are zero or in the future. I will mark these as missing before filling ages.
- **Ratings per user and book.** The median is one record for both users and ISBNs with any activity. I will compare minimum counts in Silver to see how much data remains.
- **Matrix sparsity.** Only 0.001363% of possible pairs between catalog users and books have a recorded rating. I will filter out low-activity users and books in Silver and store only the observed values in a sparse Gold matrix.

## 3. Silver

I use 10 explicit ratings as my first cutoff to test. The table below compares it with 5 and 20. I repeat the filter because dropping a book can bring a user below the cutoff, and dropping a user can do the same to a book.

In [4]:
users = raw_users.drop(columns="ingestion_timestamp").drop_duplicates().copy()
books = raw_books.drop(columns="ingestion_timestamp").drop_duplicates().copy()
ratings = raw_ratings.drop(columns="ingestion_timestamp").drop_duplicates().copy()

age = pd.to_numeric(users["Age"], errors="coerce")
age = age.where(age.between(5, 100))
users["Age"] = age.fillna(age.median())
location = users.pop("Location").str.split(",", n=2, expand=True)
location.columns = ["City", "State", "Country"]
for column in location:
    users[column] = location[column].str.strip().str.lower().replace("", pd.NA)

year = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
books["Year-Of-Publication"] = year.where(year.ne(0) & year.le(current_year)).astype("Int64")
books = books.drop(columns=["Image-URL-S", "Image-URL-M", "Image-URL-L"])
for column in ["Book-Title", "Book-Author", "Publisher"]:
    books[column] = books[column].str.strip().str.title().replace(["", "Null"], pd.NA)
books["ISBN"] = books["ISBN"].str.strip().str.upper()
ratings["ISBN"] = ratings["ISBN"].str.strip().str.upper()

users = users.drop_duplicates(subset="User-ID")
books = books.drop_duplicates(subset="ISBN")
ratings = ratings.loc[
    ratings["Book-Rating"].between(1, 10)
    & ratings["User-ID"].isin(users["User-ID"])
    & ratings["ISBN"].isin(books["ISBN"])
]
# Keep one value per user–book pair if any ratings disagree.
ratings = ratings.groupby(["User-ID", "ISBN"], as_index=False)["Book-Rating"].mean()

def filter_ratings(frame, minimum):
    while True:
        user_count = frame.groupby("User-ID")["ISBN"].transform("size")
        book_count = frame.groupby("ISBN")["User-ID"].transform("size")
        kept = frame.loc[user_count.ge(minimum) & book_count.ge(minimum)]
        if len(kept) == len(frame):
            return kept
        frame = kept

cutoffs = {minimum: filter_ratings(ratings, minimum) for minimum in [5, 10, 20]}
summary = pd.DataFrame([
    {"minimum": minimum, "users": data["User-ID"].nunique(),
     "books": data["ISBN"].nunique(), "ratings": len(data)}
    for minimum, data in cutoffs.items()
])
display(summary)

,minimum,users,books,ratings
0,5,6858,9092,115332
1,10,1820,2030,41469
2,20,0,0,0


I kept the cutoff at **10 explicit ratings per user and per book**. A cutoff of 5 keeps more of the catalog but allows books and users with as few as five ratings. A cutoff of 20 leaves no data after filtering both sides. At 10, I have 1,820 users, 2,030 books and 41,469 ratings.

In [5]:
minimum = 10
ratings = cutoffs[minimum]
users = users.loc[users["User-ID"].isin(ratings["User-ID"])]
books = books.loc[books["ISBN"].isin(ratings["ISBN"])]
assert ratings.groupby("User-ID").size().min() >= minimum
assert ratings.groupby("ISBN").size().min() >= minimum

for name, frame in [("users", users), ("books", books), ("ratings", ratings)]:
    frame.to_parquet(f"silver/{name}.parquet", index=False)
    print(name, frame.shape)

users (1820, 5)
books (2030, 5)
ratings (41469, 3)


## 4. Gold

In [6]:
ratings = pd.read_parquet("silver/ratings.parquet")
books = pd.read_parquet("silver/books.parquet")

user_ids = pd.Index(sorted(ratings["User-ID"].unique()), name="User-ID")
book_ids = pd.Index(sorted(ratings["ISBN"].unique()), name="ISBN")
matrix = csr_matrix(
    (ratings["Book-Rating"], (user_ids.get_indexer(ratings["User-ID"]),
                              book_ids.get_indexer(ratings["ISBN"]))),
    shape=(len(user_ids), len(book_ids)),
)
save_npz("gold/user_book_ratings.npz", matrix)
user_ids.to_frame(index=False).to_parquet("gold/user_ids.parquet", index=False)
lookup = books.set_index("ISBN").loc[book_ids, ["Book-Title", "Book-Author"]].reset_index()
lookup.to_parquet("gold/book_lookup.parquet", index=False)

for name, key in [("user", "User-ID"), ("book", "ISBN")]:
    stats = ratings.groupby(key)["Book-Rating"].agg(average_rating="mean", rating_count="size")
    stats.reset_index().to_parquet(f"gold/{name}_stats.parquet", index=False)

print("User–book matrix:", matrix.shape, "| Recorded ratings:", matrix.nnz)

User–book matrix: (1820, 2030) | Recorded ratings: 41469


## 5. Item-based collaborative filtering

I treat a rating of 8 or more as a favorite. For each favorite, nearby books get cosine similarity multiplied by the favorite's rating. I add the scores when multiple favorites point to the same book. Books with any recorded interaction for that user are excluded, including zero-rating records and records filtered out in Silver. When a title matches more than one ISBN, the similar-books function uses the edition with the most Silver ratings.

In [7]:
user_book = load_npz("gold/user_book_ratings.npz").tocsr()
book_user = user_book.T.tocsr()
user_ids = pd.Index(pd.read_parquet("gold/user_ids.parquet")["User-ID"])
lookup = pd.read_parquet("gold/book_lookup.parquet")
titles = lookup["Book-Title"].fillna(lookup["ISBN"])
book_counts = book_user.getnnz(axis=1)
model = NearestNeighbors(metric="cosine", algorithm="brute").fit(book_user)

history = raw_ratings.loc[raw_ratings["User-ID"].isin(user_ids), ["User-ID", "ISBN"]].copy()
history["ISBN"] = history["ISBN"].str.strip().str.upper()
seen = history.groupby("User-ID")["ISBN"].agg(set).to_dict()

def neighbors(book_index, count):
    distances, indices = model.kneighbors(
        book_user[book_index], n_neighbors=min(count + 1, len(lookup))
    )
    matches = [(int(i), 1 - float(d)) for i, d in zip(indices[0], distances[0])
               if i != book_index]
    return matches[:count]

def similar_books(title):
    matches = np.flatnonzero(titles.str.casefold().eq(title.strip().casefold()))
    if len(matches) == 0:
        return []
    book_index = matches[book_counts[matches].argmax()]
    return [titles.iloc[i] for i, _ in neighbors(book_index, 10)]

def recommend(user_id, n=10):
    if user_id not in user_ids or n <= 0:
        return []
    row = user_book[user_ids.get_loc(user_id)]
    already_seen = seen.get(user_id, set())
    seen_titles = set(titles[lookup["ISBN"].isin(already_seen)].str.casefold())
    scores = {}
    for book_index, rating in zip(row.indices, row.data):
        if rating < 8:
            continue
        for candidate, similarity in neighbors(book_index, 30):
            title = titles.iloc[candidate]
            if similarity > 0 and lookup["ISBN"].iloc[candidate] not in already_seen and title.casefold() not in seen_titles:
                scores[title] = scores.get(title, 0) + rating * similarity
    return sorted(scores, key=lambda title: (-scores[title], title))[:n]

## 6. Demo and sanity check

In [8]:
example_title = titles.iloc[book_counts.argmax()]
print("10 books similar to", example_title)
display(pd.DataFrame({"Similar book": similar_books(example_title)}))
assert len(similar_books(example_title)) == 10

# Pick three users with several favorites in the model.
favorite_counts = (user_book >= 8).getnnz(axis=1)
demo_users = user_ids[np.argsort(-favorite_counts, kind="stable")[:3]]

for user_id in demo_users:
    row = user_book[user_ids.get_loc(user_id)]
    favorites = sorted(
        [(titles.iloc[i], rating) for i, rating in zip(row.indices, row.data) if rating >= 8],
        key=lambda item: (-item[1], item[0])
    )
    recommendations = recommend(user_id)
    print(f"User {user_id} | Showing up to 10 of {len(favorites)} favorites")
    display(pd.DataFrame({
        "Highly rated book": pd.Series([f"{title} ({rating:g}/10)" for title, rating in favorites[:10]]),
        "Recommended book": pd.Series(recommendations),
    }))
    seen_titles = set(titles[lookup["ISBN"].isin(seen[user_id])].str.casefold())
    assert len(recommendations) == 10
    assert not seen_titles.intersection(title.casefold() for title in recommendations)

10 books similar to The Lovely Bones: A Novel
User 11676 | Showing up to 10 of 607 favorites
User 104636 | Showing up to 10 of 173 favorites
User 153662 | Showing up to 10 of 163 favorites


,Similar book
0,Where The Heart Is (Oprah'S Book Club (Paperback))
1,Lucky : A Memoir
2,The Da Vinci Code
3,The Pilot'S Wife : A Novel
4,Good In Bed
5,Nights In Rodanthe
6,The Pact: A Love Story
7,The Lake House
8,Lucky
9,White Oleander : A Novel (Oprah'S Book Club)


,Highly rated book,Recommended book
0,1St To Die: A Novel (10/10),"Night Shield (Intimate Moments, 1027)"
1,84 Charing Cross Road (10/10),Birthright
2,A Cold Heart: An Alex Delaware Novel (10/10),Public Secrets
3,A Man Named Dave: A Story Of Triumph And Forgiveness (10/10),A Perfect Stranger
4,A Painted House (10/10),Night Tales
5,A Pirate Looks At Fifty (10/10),Tears Of The Moon (Irish Trilogy)
6,A Prayer For Owen Meany (10/10),"The Drawing Of The Three (The Dark Tower, Book 2)"
7,A Walk To Remember (10/10),The Dark Half
8,A Wrinkle In Time (10/10),Rose Madder
9,"Ahab'S Wife: Or, The Star-Gazer: A Novel (10/10)","The Perfect Neighbor (The Macgregors) (The Macgregors : Special Edition Series, No 1232)"


,Highly rated book,Recommended book
0,1St To Die: A Novel (10/10),The Gift
1,A Knight In Shining Armor (10/10),If There Be Thorns (Dollanganger)
2,A Time To Kill (10/10),Garden Of Shadows (Dollanger Saga (Paperback))
3,Anne Of Green Gables (Anne Of Green Gables Novels (Paperback)) (10/10),The Face Of Fear
4,Anne Of Ingleside (Anne Of Green Gables Novels (Paperback)) (10/10),Seeds Of Yesterday (Dollanganger)
5,Anne'S House Of Dreams (10/10),Fallen Hearts
6,Bridge To Terabithia (10/10),Five Days In Paris
7,Charlotte'S Web (Trophy Newbery) (10/10),The Endearment
8,Eyes Of A Child (10/10),N Is For Noose (Kinsey Millhone Mysteries (Hardcover))
9,Harry Potter And The Chamber Of Secrets (Book 2) (10/10),Genuine Lies


,Highly rated book,Recommended book
0,2010: Odyssey Two (10/10),The Villa
1,A Cry In The Night (10/10),Loyalty In Death
2,A Knight In Shining Armor (10/10),Born In Shame
3,A Separate Peace (10/10),The Macgregors: Serena - Caine (The Macgregors)
4,A Thin Dark Line (Mysteries &Amp; Horror) (10/10),Ceremony In Death (Eve Dallas Mysteries (Paperback))
5,A Time To Kill (10/10),Paradise
6,Anne Of Green Gables (Anne Of Green Gables Novels (Paperback)) (10/10),Rainbow Valley (Anne Of Green Gables Novels (Paperback))
7,Anne Of The Island (10/10),The Macgregors: Alan - Grant (The Macgregors)
8,Anne Of Windy Poplars (Anne Of Green Gables Novels (Paperback)) (10/10),Defy Not The Heart
9,Betrayal In Death (10/10),Immortal In Death
